# NB-02: Cross-Reference & Label Audit

Extracts all `\\label` and `\\ref` commands, checks for undefined refs, duplicate labels, labels inside `\\item` blocks, and Supp A filename mismatches.

In [1]:
import collections
import re
from pathlib import Path

TEX_FILE = 'jmlr-hypatiax-paper-final.tex'
source = Path(TEX_FILE).read_text(encoding='utf-8')
lines  = source.splitlines()
LABEL_RE = re.compile(r'\\label\{([^}]+)\}')
REF_RE   = re.compile(r'\\(?:ref|eqref)\{([^}]+)\}')
BIBITEM_RE = re.compile(r'\\bibitem(?:\[[^\]]*\])?\{([^}]+)\}', re.M)
all_labels = LABEL_RE.findall(source)
all_refs   = REF_RE.findall(source)
label_set  = set(all_labels)
ref_set    = set(all_refs)
print(f'Labels: {len(all_labels)} total, {len(label_set)} unique')
print(f'Refs  : {len(all_refs)} total,  {len(ref_set)} unique')


Labels: 79 total, 79 unique
Refs  : 37 total,  31 unique


## Step 1 — Duplicate label keys

In [2]:
label_counts = collections.Counter(all_labels)
dup_labels = {k: v for k, v in label_counts.items() if v > 1}
print("=" * 60)
print(f"[CRITICAL] Duplicate labels (same key defined twice): {len(dup_labels)}")
print("=" * 60)
for k, n in dup_labels.items():
    lnums = [i+1 for i, ln in enumerate(lines) if f"\\label{{{k}}}" in ln]
    print(f"  DUPLICATE  {k!r}  defined {n}x  at lines {lnums}")
if not dup_labels:
    print("  OK - no duplicate labels")

[CRITICAL] Duplicate labels (same key defined twice): 0
  OK - no duplicate labels


## Step 2 — Undefined \\ref targets

In [3]:
undefined_refs = sorted(ref_set - label_set)
print("=" * 60)
print(f"[CRITICAL] Undefined references: {len(undefined_refs)}")
print("=" * 60)
for r in undefined_refs:
    lnums = [i+1 for i, ln in enumerate(lines) if f"\\ref{{{r}}}" in ln or f"\\eqref{{{r}}}" in ln]
    print(f"  UNDEF-REF  {r!r:38s}  at lines {lnums[:6]}")
if not undefined_refs:
    print("  OK - all refs have corresponding labels")

[CRITICAL] Undefined references: 0
  OK - all refs have corresponding labels


## Step 3 — Labels inside \\item blocks

In [4]:
# Detect \\label inside \\item (produces garbled \\ref output)
item_label_pat = re.compile(
    r'\\item\b.*?\\label\{([^}]+)\}', re.DOTALL)
# Simple approach: line-by-line, flag lines with \item AND \label on same line
# Also scan \\item blocks (multi-line)
SECTION_LIKE = re.compile(r'\\(?:section|subsection|subsubsection|chapter)\{')

suspicious = []
in_item = False
item_lines = []
for i, ln in enumerate(lines):
    stripped = ln.strip()
    if stripped.startswith(r'\item'):
        in_item = True
        item_lines = [(i+1, stripped)]
    elif in_item:
        item_lines.append((i+1, stripped))
        if SECTION_LIKE.search(stripped) or stripped.startswith(r'\end{enumerate}') or stripped.startswith(r'\end{itemize}'):
            in_item = False
            item_lines = []
    if in_item:
        lbl_m = re.search(r'\\label\{([^}]+)\}', stripped)
        if lbl_m:
            suspicious.append((i+1, lbl_m.group(1), stripped))

print("=" * 60)
print(f"[WARN] Labels inside \\\\item blocks (\\\\ref will produce wrong output): {len(suspicious)}")
print("=" * 60)
for lineno, key, ctx in suspicious:
    print(f"  ITEM-LABEL  line {lineno}  {key!r}  context: {ctx[:80]}")
if not suspicious:
    print("  OK - no \\label inside \\item detected")

[WARN] Labels inside \\item blocks (\\ref will produce wrong output): 2
  ITEM-LABEL  line 601  'sec:r2_bugfix'  context: \item \textbf{Inconsistent formula evaluation}\label{sec:r2_bugfix}: Earlier
  ITEM-LABEL  line 1094  'thm:five_system_hierarchy'  context: \label{thm:five_system_hierarchy}


## Step 4 — Known structural cross-reference issues

In [5]:
# Known structural cross-reference issues (pre-audited)
known_issues = [
    ("sec:llm_limitations", "sec:llm_domain",
     "Both applied to Section 3 — duplicate section labels.  Keep only one."),
    ("sec:r2_bugfix",
     "Applied to \\\\item inside enumerate (Section 6.5 corrections). "
     "\\\\ref{sec:r2_bugfix} will not produce a section number.  "
     "Move label to the \\\\subsection above, or replace with \\\\nameref."),
]
for issue in known_issues:
    if len(issue) == 3:
        k1, k2, note = issue
        lnums1 = [i+1 for i, ln in enumerate(lines) if f"\\label{{{k1}}}" in ln]
        lnums2 = [i+1 for i, ln in enumerate(lines) if f"\\label{{{k2}}}" in ln]
        print(f"KNOWN ISSUE: {k1!r} (line {lnums1}) and {k2!r} (line {lnums2})")
        print(f"  Note: {note}\n")
    else:
        k1, note = issue
        lnums = [i+1 for i, ln in enumerate(lines) if k1 in ln]
        print(f"KNOWN ISSUE: {k1!r}  at lines {lnums}")
        print(f"  Note: {note}\n")

KNOWN ISSUE: 'sec:llm_limitations' (line [325]) and 'sec:llm_domain' (line [325])
  Note: Both applied to Section 3 — duplicate section labels.  Keep only one.

KNOWN ISSUE: 'sec:r2_bugfix'  at lines [601, 902]
  Note: Applied to \\item inside enumerate (Section 6.5 corrections). \\ref{sec:r2_bugfix} will not produce a section number.  Move label to the \\subsection above, or replace with \\nameref.



## Step 5 — Supplementary A filename / section references

In [6]:
# Check Supplementary A filename references inside main paper
# The actual filename is jmlr-hypatiax-paper-final.tex
# Supp A may reference 'jmlr_paper_main.tex' (wrong)
bad_refs = []
for i, ln in enumerate(lines):
    if "jmlr_paper_main" in ln or "jmlr_paper_final" in ln:
        bad_refs.append((i+1, ln.strip()))
print("=" * 60)
print("Filename references in main paper:")
print("=" * 60)
if bad_refs:
    for lno, ctx in bad_refs:
        print(f"  line {lno}: {ctx[:100]}")
else:
    print("  None found (filename refs likely only in Supp A)")

# Cross-reference: Supp A says 'Section 7.3 (Component 3)'
# but in main paper it is Section 7.4 (Five-Stage Routing)
print()
print("Manual check required:")
print("  Supp A references 'Section 7.3 (Component 3)' for Proposition 1.")
print("  In main paper it is Section 7.4 (Component 3: Five-Stage Routing).")
print("  -> Fix: change '7.3' to '7.4' in supp_routing_improvements.tex")

Filename references in main paper:
  line 323: % === SECTION: Empirical Evidence (from jmlr_paper_final.tex §3) ===
  line 365: % === SECTION: Theoretical Framework (from jmlr_paper_final.tex §4) ===

Manual check required:
  Supp A references 'Section 7.3 (Component 3)' for Proposition 1.
  In main paper it is Section 7.4 (Component 3: Five-Stage Routing).
  -> Fix: change '7.3' to '7.4' in supp_routing_improvements.tex


## Step 6 — Fix recipe

In [7]:
fixes = (
    "FIX-XR1  Duplicate section labels sec:llm_limitations / sec:llm_domain\n"
    "  Both are on Section 3.  Remove sec:llm_domain; update any \\ref{sec:llm_domain}\n"
    "  to \\ref{sec:llm_limitations}.\n\n"
    "FIX-XR2  \\label{sec:r2_bugfix} is inside \\item, not a \\section\n"
    "  It lives in Section 6.5 item 3.  \\ref{sec:r2_bugfix} -> garbled output.\n"
    "  ACTION: Move \\label to the \\subsection{Pipeline Corrections in v3.0} heading\n"
    "  or replace \\ref{sec:r2_bugfix} with \\ref{sec:corrections} in all call sites.\n\n"
    "FIX-XR3  Supp A references Section 7.3 (Component 3)\n"
    "  Main paper has Component 3 in Section 7.4.  Fix '7.3' -> '7.4' in\n"
    "  supp_routing_improvements.tex.\n\n"
    "FIX-XR4  Supp A filename reference\n"
    "  Supp A may say 'jmlr_paper_main.tex'; actual filename is\n"
    "  'jmlr-hypatiax-paper-final.tex'.  Update all filename strings in Supp A.\n"
)
print(fixes)

FIX-XR1  Duplicate section labels sec:llm_limitations / sec:llm_domain
  Both are on Section 3.  Remove sec:llm_domain; update any \ref{sec:llm_domain}
  to \ref{sec:llm_limitations}.

FIX-XR2  \label{sec:r2_bugfix} is inside \item, not a \section
  It lives in Section 6.5 item 3.  \ref{sec:r2_bugfix} -> garbled output.
  ACTION: Move \label to the \subsection{Pipeline Corrections in v3.0} heading
  or replace \ref{sec:r2_bugfix} with \ref{sec:corrections} in all call sites.

FIX-XR3  Supp A references Section 7.3 (Component 3)
  Main paper has Component 3 in Section 7.4.  Fix '7.3' -> '7.4' in
  supp_routing_improvements.tex.

FIX-XR4  Supp A filename reference
  Supp A may say 'jmlr_paper_main.tex'; actual filename is
  'jmlr-hypatiax-paper-final.tex'.  Update all filename strings in Supp A.

